In [0]:
WITH base AS (
    SELECT 
        A.*,
        B.UserID,

        COALESCE(NULLIF(B.Gender, 'None'), 'Unknown') AS Gender,
        COALESCE(NULLIF(B.Race, 'None'), 'Unknown') AS Race,
        COALESCE(NULLIF(B.Province, 'None'), 'Unknown') AS Province,

        CASE WHEN B.Age = 0 THEN NULL ELSE B.Age END AS Age,

        -- Convert UTC to SA TIME
        from_utc_timestamp(A.RecordDate2, 'Africa/Johannesburg') AS `SA TIME`,

        DAYNAME(A.RecordDate2) AS `Day Name`,
        date_format(A.RecordDate2, 'MMMM') AS `Month Name`,

        -- Viewership Duration seconds
        hour(A.Duration2)*3600 +
        minute(A.Duration2)*60 +
        second(A.Duration2) AS `Viewership Duration seconds`

    FROM workspace.bright_tv.viewership A
    JOIN workspace.bright_tv.user_profiles B
      ON A.UserID0 = B.UserID
)

SELECT 
    Channel2,
    Gender,
    Race,
    Province,
    `Month Name`,
    `Day Name`,
   `Viewership Duration seconds`,


    -- Time bucket
    CASE
        WHEN HOUR(`SA TIME`) BETWEEN 0 AND 4 THEN 'Early Morning'
        WHEN HOUR(`SA TIME`) BETWEEN 5 AND 8 THEN 'Morning Rush'
        WHEN HOUR(`SA TIME`) BETWEEN 9 AND 11 THEN 'Mid Morning'
        WHEN HOUR(`SA TIME`) BETWEEN 12 AND 14 THEN 'Mid Day'
        WHEN HOUR(`SA TIME`) BETWEEN 15 AND 16 THEN 'Afternoon'
        WHEN HOUR(`SA TIME`) BETWEEN 17 AND 18 THEN 'Evening'
        WHEN HOUR(`SA TIME`) BETWEEN 19 AND 20 THEN 'Prime Time'
        ELSE 'Night'
    END AS Time_Bucket,

    -- Age bucket
    CASE 
        WHEN Age BETWEEN 1 AND 12 THEN 'Kids'
        WHEN Age BETWEEN 13 AND 19 THEN 'Teens'
        WHEN Age BETWEEN 20 AND 40 THEN 'Young Adults'
        WHEN Age > 40 THEN 'Adults'
        ELSE 'Unknown'
    END AS Age_Group,

    -- Metrics
    AVG(`Viewership Duration seconds`) AS Avg_Streaming_Duration,
    COUNT(*) AS Total_Streams

FROM base
GROUP BY 
    Channel2, 
    Gender, 
    Race, 
    Province, 
    `Month Name`,
    `Day Name`,
    `Viewership Duration seconds`,

    -----time bucket expression
    CASE
        WHEN HOUR(`SA TIME`) BETWEEN 0 AND 4 THEN 'Early Morning'
        WHEN HOUR(`SA TIME`) BETWEEN 5 AND 8 THEN 'Morning Rush'
        WHEN HOUR(`SA TIME`) BETWEEN 9 AND 11 THEN 'Mid Morning'
        WHEN HOUR(`SA TIME`) BETWEEN 12 AND 14 THEN 'Mid Day'
        WHEN HOUR(`SA TIME`) BETWEEN 15 AND 16 THEN 'Afternoon'
        WHEN HOUR(`SA TIME`) BETWEEN 17 AND 18 THEN 'Evening'
        WHEN HOUR(`SA TIME`) BETWEEN 19 AND 20 THEN 'Prime Time'
        ELSE 'Night'
    END,

    CASE 
        WHEN Age BETWEEN 1 AND 12 THEN 'Kids'
        WHEN Age BETWEEN 13 AND 19 THEN 'Teens'
        WHEN Age BETWEEN 20 AND 40 THEN 'Young Adults'
        WHEN Age > 40 THEN 'Adults'
        ELSE 'Unknown'
    END

ORDER BY Total_Streams DESC;